In [0]:
CATALOG = 'SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDERITEMS_ENRICHED'

In [0]:
orderitems_df = spark.table('SALES.BRONZE.orderitems')
orderitems_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

orderitems_enriched_df = orderitems_df.select(
trim(col("order_id")).alias("order_id"),

col("order_item_id").cast("int").alias("order_item_id"),

trim(col("product_id")).alias("product_id"),

trim(col("seller_id")).alias("seller_id"),

to_timestamp(col("shipping_limit_date")).alias("shipping_limit_date"),

col("price").cast("double").alias("price"),

col("freight_value").cast("double").alias("freight_value"),

col("file_path").alias("file_path"),

current_timestamp().alias('load_timestamp')

).filter('order_id IS NOT NULL')

display(orderitems_enriched_df.limit(1))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


w = ( Window.partitionBy("order_id", "order_item_id") .orderBy(col("shipping_limit_date").desc_nulls_last(), col("price").desc_nulls_last()) )

In [0]:
nodup_df = orderitems_enriched_df.withColumn("row_num", row_number().over(w))

In [0]:
flagged_df = (nodup_df.select("order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value", "file_path","load_timestamp") .withColumn("is_price_non_positive", col("price") <= 0) .withColumn("is_freight_negative", col("freight_value") <= 0) .withColumn("is_ship_date_outlier", (year(col("shipping_limit_date")) < lit(2017)) | (year(col("shipping_limit_date")) > lit(2019))) )

In [0]:
display(flagged_df.limit(1))

In [0]:
flagged_df.createOrReplaceTempView('enriched_orderitems_temp_vw')

In [0]:
%sql
select * from enriched_orderitems_temp_vw 

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_orderitems_temp_vw
  )
  WHERE rn = 1
) src
ON (src.order_id  = tgt.order_id)
WHEN MATCHED THEN
  UPDATE SET
    tgt.order_id = src.order_id,
    tgt.order_item_id = src.order_item_id,
    tgt.product_id = src.product_id,
    tgt.seller_id = src.seller_id,
    tgt.shipping_limit_date = src.shipping_limit_date,
    tgt.price = src.price,
    tgt.freight_value = src.freight_value
    
WHEN NOT MATCHED THEN
  INSERT (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value,
    origin_file_path,
    is_price_non_positive,
    is_freight_negative,
    is_ship_date_outlier,
    load_timestamp
  )
  VALUES (
    src.order_id,
    src.order_item_id,
    src.product_id,
    src.seller_id,
    src.shipping_limit_date,
    src.price,
    src.freight_value,
    src.file_path,
    src.is_price_non_positive,
    src.is_freight_negative,
    src.is_ship_date_outlier,
    current_timestamp()
  )
""")

In [0]:
%sql
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_orderitems_temp_vw
  )
  WHERE rn = 1